# Analyse des réservations d'hôtels

## Questions métier

1. Quel est le taux d'annulation, et varie-t-il selon l'hôtel et le type de dépôt ?
2. Le délai de réservation influence-t-il les annulations ?
3. Quels mois sont les plus chargés, et quel est le prix moyen par nuit selon la saison ?
4. D'où viennent les clients, et quels pays annulent le plus ?
5. Quel segment de marché rapporte le plus ?

In [1]:
import pandas as pd

hotels = pd.read_csv(
    "../data/raw/hotel_bookings.csv",
    parse_dates=["reservation_status_date"],
    dtype={"agent": "Int64", "company": "Int64"},
)
hotels

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,<NA>,<NA>,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,<NA>,<NA>,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,<NA>,<NA>,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304,<NA>,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240,<NA>,0,Transient,98.00,0,1,Check-Out,2015-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,394,<NA>,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,9,<NA>,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,9,<NA>,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,89,<NA>,0,Transient,104.40,0,0,Check-Out,2017-09-07


In [2]:
hotels.shape

(119390, 32)

In [3]:
hotels.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119390 non-null  str           
 1   is_canceled                     119390 non-null  int64         
 2   lead_time                       119390 non-null  int64         
 3   arrival_date_year               119390 non-null  int64         
 4   arrival_date_month              119390 non-null  str           
 5   arrival_date_week_number        119390 non-null  int64         
 6   arrival_date_day_of_month       119390 non-null  int64         
 7   stays_in_weekend_nights         119390 non-null  int64         
 8   stays_in_week_nights            119390 non-null  int64         
 9   adults                          119390 non-null  int64         
 10  children                        119386 non-null  float64       
 11

In [4]:
hotels.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,...,119390.000000,119390.000000,119390.000000,103050.0,6797.0,119390.000000,119390.000000,119390.000000,119390.000000,119390
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103890,0.007949,...,0.087118,0.137097,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363,2016-07-30 00:24:47.883407
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.0,6.0,0.000000,-6.380000,0.000000,0.000000,2014-10-17 00:00:00
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,9.0,62.0,0.000000,69.290000,0.000000,0.000000,2016-02-01 00:00:00
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,14.0,179.0,0.000000,94.575000,0.000000,0.000000,2016-08-07 00:00:00
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,229.0,270.0,0.000000,126.000000,0.000000,1.000000,2017-02-08 00:00:00
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,...,26.000000,72.000000,21.000000,535.0,543.0,391.000000,5400.000000,8.000000,5.000000,2017-09-14 00:00:00
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398561,0.097436,...,0.844336,1.497437,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798,NaN


In [5]:
hotels.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [6]:
hotels.duplicated().sum()

np.int64(31994)

## Exploration

Filtres, tris, comptages et distributions sur les données brutes. Chaque constat devient une décision de nettoyage, écrite dans `02_nettoyage.ipynb`.

In [7]:
annulees = hotels[hotels["is_canceled"] == 1]
print(f"Réservations annulées : {len(annulees)}")

Réservations annulées : 44224


In [8]:
hotels["nuits"] = hotels["stays_in_weekend_nights"] + hotels["stays_in_week_nights"]
longs_sejours = hotels[hotels["nuits"] >= 14]
print(f"Séjours de 14 nuits ou plus : {len(longs_sejours)}")
longs_sejours[["hotel", "nuits", "adr", "is_canceled"]].head(10)

Séjours de 14 nuits ou plus : 1355


,hotel,nuits,adr,is_canceled
30,Resort Hotel,14,62.00,0
31,Resort Hotel,15,63.86,0
52,Resort Hotel,14,110.70,0
85,Resort Hotel,14,134.73,0
86,Resort Hotel,14,92.45,0
87,Resort Hotel,21,108.73,1
158,Resort Hotel,14,74.07,0
159,Resort Hotel,14,134.10,0
190,Resort Hotel,14,110.70,0
191,Resort Hotel,14,93.60,0


In [9]:
familles = hotels[(hotels["children"] > 0) | (hotels["babies"] > 0)]
print(f"Réservations avec enfants ou bébés : {len(familles)}")

Réservations avec enfants ou bébés : 9332


In [10]:
ville_tres_en_avance = hotels[(hotels["hotel"] == "City Hotel") & (hotels["lead_time"] > 365)]
print(f"Hôtel de ville, réservé plus d'un an à l'avance : {len(ville_tres_en_avance)}")

Hôtel de ville, réservé plus d'un an à l'avance : 2704


In [11]:
hotels.loc[hotels["adr"] > 1000, ["hotel", "adr", "market_segment", "is_canceled"]]

,hotel,adr,market_segment,is_canceled
48515,City Hotel,5400.0,Offline TA/TO,1


In [12]:
hotels.sort_values("adr", ascending=False).head(5)[
    ["hotel", "arrival_date_year", "market_segment", "adr", "is_canceled"]
]

,hotel,arrival_date_year,market_segment,adr,is_canceled
48515,City Hotel,2016,Offline TA/TO,5400.0,1
111403,City Hotel,2017,Offline TA/TO,510.0,0
15083,Resort Hotel,2015,Corporate,508.0,0
103912,City Hotel,2016,Direct,451.5,0
13142,Resort Hotel,2017,Online TA,450.0,1


In [13]:
hotels.sort_values("adr").head(3)[
    ["hotel", "arrival_date_year", "market_segment", "adr", "is_canceled"]
]

,hotel,arrival_date_year,market_segment,adr,is_canceled
14969,Resort Hotel,2017,Groups,-6.38,0
60651,City Hotel,2016,Corporate,0.00,1
94383,City Hotel,2017,Complementary,0.00,0


In [14]:
hotels.sort_values("lead_time", ascending=False).head(3)[["hotel", "lead_time", "is_canceled"]]

,hotel,lead_time,is_canceled
1,Resort Hotel,737,0
4182,Resort Hotel,709,0
65241,City Hotel,629,1


In [15]:
hotels.sort_values("nuits", ascending=False).head(3)[["hotel", "nuits", "adr"]]

,hotel,nuits,adr
14038,Resort Hotel,69,110.00
14037,Resort Hotel,60,110.50
101794,City Hotel,57,8.34


In [16]:
for colonne in ["hotel", "market_segment", "deposit_type", "meal", "customer_type"]:
    print(colonne)
    display((hotels[colonne].value_counts(normalize=True) * 100).round(1))

hotel


hotel
City Hotel      66.4
Resort Hotel    33.6
Name: proportion, dtype: float64

market_segment


market_segment
Online TA        47.3
Offline TA/TO    20.3
Groups           16.6
Direct           10.6
Corporate         4.4
Complementary     0.6
Aviation          0.2
Undefined         0.0
Name: proportion, dtype: float64

deposit_type


deposit_type
No Deposit    87.6
Non Refund    12.2
Refundable     0.1
Name: proportion, dtype: float64

meal


meal
BB           77.3
HB           12.1
SC            8.9
Undefined     1.0
FB            0.7
Name: proportion, dtype: float64

customer_type


customer_type
Transient          75.1
Transient-Party    21.0
Contract            3.4
Group               0.5
Name: proportion, dtype: float64

In [17]:
hotels["lead_time"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(1)

count    119390.0
mean        104.0
std         106.9
min           0.0
10%           3.0
25%          18.0
50%          69.0
75%         160.0
90%         265.0
99%         444.0
max         737.0
Name: lead_time, dtype: float64

In [18]:
hotels["adr"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(1)

count    119390.0
mean        101.8
std          50.5
min          -6.4
1%            0.0
25%          69.3
50%          94.6
75%         126.0
99%         252.0
max        5400.0
Name: adr, dtype: float64

In [19]:
hotels["nuits"].describe(percentiles=[0.5, 0.9, 0.99]).round(1)

count    119390.0
mean          3.4
std           2.6
min           0.0
50%           3.0
90%           7.0
99%          14.0
max          69.0
Name: nuits, dtype: float64

In [20]:
hotels.groupby("hotel")["adr"].median().round(1)

hotel
City Hotel      99.9
Resort Hotel    75.0
Name: adr, dtype: float64

In [21]:
sans_client = (hotels["adults"] + hotels["children"].fillna(0) + hotels["babies"]) == 0
print(f"Réservations sans client : {sans_client.sum()}")
print(f"Réservations sans nuit : {(hotels['nuits'] == 0).sum()}")
print(f"Prix à 0 : {(hotels['adr'] == 0).sum()} / prix négatifs : {(hotels['adr'] < 0).sum()}")

Réservations sans client : 180
Réservations sans nuit : 715
Prix à 0 : 1959 / prix négatifs : 1


### Constats d'exploration et décisions de nettoyage

| Constat | Décision (voir `02_nettoyage.ipynb`) |
|---|---|
| un prix de 5400 alors que la médiane est de 94.6, et un prix négatif (-6.38) | erreurs de saisie : lignes supprimées |
| 180 réservations sans client, 715 sans nuit | incohérentes : supprimées |
| 1959 prix à 0 (1,6 % des réservations), dont des séjours offerts | conservés, mais exclus des moyennes de prix |
| 31994 doublons | supprimés, avec une limite notée sur les réservations de groupes |
| délais jusqu'à 737 jours | plausibles : conservés |
| « Undefined » à 1,0 % dans `meal` | regroupé avec « SC » (aucune formule repas) |

**Ce que l'exploration apprend :** l'hôtel de ville représente 66.4 % des réservations et l'agence en ligne (Online TA) 47.3 %. La grande majorité des clients réservent sans dépôt (87.6 %), en séjour individuel (Transient, 75.1 %). Le délai de réservation est très étalé (médiane de 69 jours, un quart des réservations à moins de 18 jours). Le séjour médian dure 3 nuits et le prix médian par nuit est de 94.6, plus élevé à l'hôtel de ville (99.9) qu'à l'hôtel de villégiature (75.0).